# Add documentation
This notebook contains a helper function to set table description and column descriptions in a structured markdown format. Then the function is used to set table descriptions and column descriptions on some entities.

##Helper functions

In [0]:
# Databricks / PySpark
from typing import Dict, Optional, Literal
import re

def _q(s: str) -> str:
    """Escape single quotes for SQL string literals."""
    return s.replace("'", "''") if s else s

def _split_3part(name: str):
    """
    Split 'catalog.schema.table' (of 'schema.table' als er een USE CATALOG actief is).
    """
    parts = name.split(".")
    if len(parts) == 3:
        return parts[0], parts[1], parts[2]
    elif len(parts) == 2:
        # current catalog + schema.table
        current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
        return current_catalog, parts[0], parts[1]
    else:
        raise ValueError("Gebruik minimaal schema.tabel of catalog.schema.tabel")

def _get_table_comment(table_name: str) -> Optional[str]:
    try:
        cat, sch, tbl = _split_3part(table_name)
        return (
            spark.sql(f"""
                SELECT comment
                FROM {cat}.information_schema.tables
                WHERE table_schema = '{_q(sch)}' AND table_name = '{_q(tbl)}'
            """).collect()[0][0]
        )
    except Exception:
        return None

def update_structured_comments_md(
    table_name: str,
    *,
    description_md: Optional[str] = None,
    column_comments_md: Optional[Dict[str, str]] = None,
    mode: Literal["replace","append"] = "replace",  # append plakt onderaan bestaande tekst
    append_separator: str = "\n\n---\n\n"
):
    """
    Zet Markdown-beschrijvingen op een UC-tabel en kolommen.
    Vereist: USE CATALOG/SCHEMA + MODIFY op de tabel.
    """
    # 1) Tabelbeschrijving
    if description_md:
        if mode == "append":
            existing = _get_table_comment(table_name)
            if existing:
                description_md = f"{existing}{append_separator}{description_md}"
        spark.sql(f"COMMENT ON TABLE {table_name} IS '{_q(description_md)}'")

    # 2) Kolomcommentaren
    if column_comments_md:
        # Valideer kolommen tegen information_schema (optioneel maar handig)
        cat, sch, tbl = _split_3part(table_name)
        existing_cols = {
            r[0].lower()
            for r in spark.sql(f"""
                SELECT column_name
                FROM {cat}.information_schema.columns
                WHERE table_schema='{_q(sch)}' AND table_name='{_q(tbl)}'
            """).collect()
        }
        for col, md in column_comments_md.items():
            if col.lower() not in existing_cols:
                raise ValueError(f"Kolom '{col}' bestaat niet in {table_name}")
            if mode == "append":
                # bestaand kolomcomment ophalen
                try:
                    cur = spark.sql(f"""
                        SELECT comment FROM {cat}.information_schema.columns
                        WHERE table_schema='{_q(sch)}'
                          AND table_name='{_q(tbl)}'
                          AND column_name='{_q(col)}'
                    """).collect()[0][0]
                except Exception:
                    cur = None
                if cur:
                    md = f"{cur}{append_separator}{md}"
            spark.sql(f"COMMENT ON COLUMN {table_name}.{col} IS '{_q(md)}'")

    return {"table": table_name, "mode": mode, "updated": True}


##Set the table descriptions and column descriptions

#### _For the dim_employee table_

In [0]:
table_md = """\
**Owner:** `HRD`  
**Slack Channel:** #hrd  
**Emails:** hrd@minfosupport.com

### Description
Employee information, including personal details and employment data. It can be used for various purposes such as analyzing workforce demographics, tracking employee tenure, and understanding departmental structures. Key data points include employee names, titles, hire dates, and contact information.

> Data is updated weekly. Timestamps are in **UTC**.

**SLA:** 6 hours
"""

col_md = {
        "EmployeeID": "Unique identifier for each employee",
        "FirstName": "First name of the employee",
        "LastName": "Last name of the employee",
        "Gender": "Gender of the employee",
        "BirthDate": "Date of birth of the employee",
        "HireDate": "Date when the employee was hired"
}

update_structured_comments_md(
    table_name="workspace.adventureworks.dim_employee",
    description_md=table_md,
    column_comments_md=col_md,
    mode="replace",  # of "append" als je wilt bijplakken
)


#### _For the fact_reseller_sales table_

In [0]:
table_md = """\
### Description
The table contains data related to sales orders, including details about the reseller, employee, product, and various financial metrics. Use cases include analyzing sales performance, tracking order fulfillment timelines, and assessing the impact of discounts and taxes on sales revenue. Key information includes order dates, shipping dates, and amounts associated with each sale.
"""

update_structured_comments_md(
    table_name="workspace.adventureworks.fact_reseller_sales",
    description_md=table_md,
    mode="replace",  # of "append" als je wilt bijplakken
)
